# Session 10 — Capstone: The Impact of ChatGPT's Public Release on Tech Firms

**Course: Event Studies in Finance & Economics**

*Mathis Mourey*

---

This final session applies the full event study toolkit developed across Sessions 1 through 9 to a single research question: did the public release of ChatGPT on November 30, 2022 generate abnormal stock returns for technology firms, and did the direction and magnitude of these returns depend on the firm's exposure to artificial intelligence?

The ChatGPT release is an attractive event study candidate for several reasons. The date is precise and publicly known. The release was largely unanticipated by financial markets: while GPT-3 had existed since 2020 and DALL-E since early 2022, the public availability of a conversational AI interface was not pre-announced through the usual channels (no SEC filing, no investor call, no press embargo). The event affected a broad cross-section of technology firms with heterogeneous exposure: chipmakers and cloud providers stood to benefit from increased AI compute demand, while firms whose business models could be disrupted by generative AI (online education, freelance marketplaces, content creation) faced a potential threat. This heterogeneity in expected impact makes the cross-sectional analysis (Section 7) particularly informative.

**References for this session:**

- Eisfeldt, A.L., Schubert, G. and Zhang, M.B. (2023). Generative AI and Firm Values. NBER Working Paper No. 31222.
- Eloundou, T., Manning, S., Mishkin, P. and Rock, D. (2023). GPTs are GPTs: An Early Look at the Labor Market Impact Potential of Large Language Models. arXiv:2303.10130.
- Fama, E.F., Fisher, L., Jensen, M.C. and Roll, R. (1969). The Adjustment of Stock Prices to New Information. *International Economic Review*, 10(1), 1--21.
- MacKinlay, A.C. (1997). Event Studies in Economics and Finance. *Journal of Economic Literature*, 35(1), 13--39.
- Noy, S. and Zhang, W. (2023). Experimental Evidence on the Productivity Effects of Generative Artificial Intelligence. *Science*, 381(6654), 187--192.

## 1. The ChatGPT Release as an Information Shock

On November 30, 2022, OpenAI released ChatGPT, a conversational interface to its GPT-3.5 language model, as a free public research preview. The product reached one million users within five days and 100 million within two months, making it the fastest-adopted consumer technology in history.

From the perspective of financial markets, the release constituted an information shock about the commercial viability of large language models. Before November 30, the capabilities of GPT-3 were known to researchers and developers, but the general public and most equity analysts had limited direct experience with the technology. The ChatGPT interface made the capabilities tangible, triggering a rapid reassessment of which firms would benefit from the AI transition and which would be disrupted.

Eisfeldt, Schubert, and Zhang (2023) documented that firms with higher AI exposure (measured by the frequency of AI-related terms in earnings calls and job postings) earned significantly higher returns in the months following the ChatGPT release. Their analysis suggests that the market repriced AI exposure rapidly but not instantaneously, with abnormal returns accruing over days to weeks rather than within a single trading session.

Our study complements this work by applying the formal event study methodology of this course: precise event windows, standard test statistics, cross-sectional regressions, and a full robustness protocol.

## 2. Sample Construction

We classify publicly traded U.S. technology firms into three groups based on their ex ante exposure to generative AI. The classification is made before examining the stock price data and is based on the firm's business model as of November 2022.

**AI Beneficiaries.** Firms whose products or infrastructure directly support AI development and deployment. This includes GPU manufacturers (NVIDIA, AMD), cloud and AI platform providers (Microsoft, Alphabet), enterprise AI firms (Palantir, C3.ai), and cloud data infrastructure firms (Snowflake, MongoDB). These firms stand to benefit from increased demand for AI compute, AI-enabled cloud services, and enterprise AI adoption.

**AI-Disrupted.** Firms whose core business model is potentially threatened by generative AI. This includes online education platforms where AI can substitute for human tutors (Chegg), freelance marketplaces where AI can replace human contractors for writing, coding, and design tasks (Upwork, Fiverr), and website builders where AI can automate web design (Wix, GoDaddy). The disruption hypothesis is that generative AI reduces demand for these firms' services by enabling users to perform the same tasks themselves.

**Control Group.** Large technology firms with ambiguous or mixed AI exposure. These firms are neither pure beneficiaries nor pure disruption targets; their AI exposure is either indirect, diversified, or offset by other business lines. This group serves as a benchmark for the overall tech sector reaction.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# -- Sample definition --
firms = pd.DataFrame({
    'ticker': [
        # AI Beneficiaries
        'NVDA', 'MSFT', 'GOOGL', 'AMD', 'PLTR', 'AI', 'SNOW', 'MDB',
        # AI-Disrupted
        'CHGG', 'UPWK', 'WIX', 'GDDY', 'FVRR',
        # Control (broad tech)
        'AAPL', 'AMZN', 'CRM', 'ADBE', 'INTC', 'IBM',
    ],
    'name': [
        'NVIDIA', 'Microsoft', 'Alphabet', 'AMD', 'Palantir', 'C3.ai', 'Snowflake', 'MongoDB',
        'Chegg', 'Upwork', 'Wix', 'GoDaddy', 'Fiverr',
        'Apple', 'Amazon', 'Salesforce', 'Adobe', 'Intel', 'IBM',
    ],
    'group': [
        'Beneficiary', 'Beneficiary', 'Beneficiary', 'Beneficiary',
        'Beneficiary', 'Beneficiary', 'Beneficiary', 'Beneficiary',
        'Disrupted', 'Disrupted', 'Disrupted', 'Disrupted', 'Disrupted',
        'Control', 'Control', 'Control', 'Control', 'Control', 'Control',
    ],
})

event_date = pd.Timestamp('2022-11-30')

print("Sample: ChatGPT Release Event Study")
print("=" * 60)
print(f"Event date: {event_date.date()} (ChatGPT public release)")
print(f"Total firms: {len(firms)}")
for g in ['Beneficiary', 'Disrupted', 'Control']:
    subset = firms[firms['group'] == g]
    print(f"\n  {g} (N={len(subset)}):")
    for _, r in subset.iterrows():
        print(f"    {r['ticker']:6s} {r['name']}")

## 3. Design Choices

All design choices are stated here before any estimation.

**Event date:** November 30, 2022.

**Event windows:** $[-1, +1]$ (primary, 3 trading days), $[0, 0]$ (event day only), $[0, +5]$ (one-week post-event), $[0, +10]$ (two-week post-event). The wider windows are motivated by the possibility that the market took time to process the implications of ChatGPT for specific firms. Eisfeldt et al. (2023) found that AI-related repricing continued for several weeks.

**Estimation window:** 250 trading days ending 10 days before the event window. This places the estimation window approximately from October 2021 to mid-November 2022, a period that includes substantial market volatility (2022 tech selloff) but no comparable AI-specific shock.

**Normal return model:** Market model with S&P 500 as benchmark (primary). Fama-French three-factor model as robustness check.

**Test statistics:** BMP test and Corrado rank test (primary), cross-sectional t-test and generalized sign test (robustness). Tests are conducted separately for each group and for the pooled sample.

**Cross-sectional regression:** CAR regressed on group dummies, log market capitalization, and market model beta.

**Confounding events:** The week of November 28 to December 2, 2022 did not contain a major macroeconomic release or Fed announcement. The November FOMC meeting was on November 1-2; the December meeting was December 13-14. The main potential confounder is general year-end portfolio rebalancing, which we control for via the market model.

In [ ]:
# -- Download data --
market_ticker = '^GSPC'
start_date = '2021-06-01'
end_date = '2023-06-01'

data_mkt = yf.download(market_ticker, start=start_date, end=end_date, progress=False)
if isinstance(data_mkt.columns, pd.MultiIndex):
    data_mkt.columns = data_mkt.columns.get_level_values(0)
mkt_ret = data_mkt['Close'].pct_change().dropna()

all_tickers = firms['ticker'].tolist()
data_firms = yf.download(all_tickers, start=start_date, end=end_date, progress=False)
if isinstance(data_firms.columns, pd.MultiIndex):
    firm_prices = data_firms['Close']
else:
    firm_prices = data_firms[['Close']].rename(columns={'Close': all_tickers[0]})
firm_ret = firm_prices.pct_change().dropna()

common = mkt_ret.index.intersection(firm_ret.index)
mkt_ret = mkt_ret.loc[common]
firm_ret = firm_ret.loc[common]
trading_days = mkt_ret.index

print(f"Data: {len(common)} trading days ({common[0].date()} to {common[-1].date()})")
print(f"Firms downloaded: {firm_ret.shape[1]} of {len(all_tickers)}")
missing = [t for t in all_tickers if t not in firm_ret.columns]
if missing:
    print(f"Missing: {missing}")

## 4. Estimation and Abnormal Returns

We estimate the market model for each firm and compute abnormal returns for all four event windows. The estimation follows the standard procedure of Sessions 2 and 3.

In [ ]:
# -- Estimate market models and compute CARs --
est_length = 250
buffer = 10
event_windows = [(-1, 1), (0, 0), (0, 5), (0, 10)]

eidx = trading_days.get_indexer([event_date], method='ffill')[0]

# Primary window AR panel for tests
primary_win = (-1, 1)
primary_days = list(range(primary_win[0], primary_win[1] + 1))
ar_panels = {}  # keyed by group
full_results = []

for _, row in firms.iterrows():
    tick = row['ticker']
    group = row['group']

    if tick not in firm_ret.columns:
        continue

    ev_s = eidx + primary_win[0]
    ev_e_max = eidx + 10  # widest window
    est_e = ev_s - buffer - 1
    est_s = est_e - est_length + 1

    if est_s < 0 or ev_e_max >= len(trading_days):
        print(f"  {tick}: insufficient data")
        continue

    y_est = firm_ret.iloc[est_s:est_e+1][tick].values
    x_est = mkt_ret.iloc[est_s:est_e+1].values
    valid = ~(np.isnan(y_est) | np.isnan(x_est))
    X = sm.add_constant(x_est[valid])
    ols = sm.OLS(y_est[valid], X).fit()
    alpha, beta = ols.params
    sigma = np.sqrt(ols.mse_resid)

    res = {'ticker': tick, 'name': row['name'], 'group': group,
           'alpha': alpha, 'beta': beta, 'sigma': sigma, 'R2': ols.rsquared}

    for tau_a, tau_b in event_windows:
        s = eidx + tau_a
        e = eidx + tau_b
        y_ev = firm_ret.iloc[s:e+1][tick].values
        x_ev = mkt_ret.iloc[s:e+1].values
        ar = y_ev - (alpha + beta * x_ev)
        res[f'CAR_[{tau_a:+d},{tau_b:+d}]'] = np.nansum(ar)

    # Store primary window ARs by group
    y_prim = firm_ret.iloc[eidx+primary_win[0]:eidx+primary_win[1]+1][tick].values
    x_prim = mkt_ret.iloc[eidx+primary_win[0]:eidx+primary_win[1]+1].values
    ar_prim = y_prim - (alpha + beta * x_prim)

    if group not in ar_panels:
        ar_panels[group] = pd.DataFrame(index=primary_days, dtype=float)
    ar_panels[group][tick] = ar_prim

    full_results.append(res)

results_df = pd.DataFrame(full_results)

# Display results by group
print("Estimated CARs by Firm and Group (%)")
print("=" * 95)
for g in ['Beneficiary', 'Disrupted', 'Control']:
    subset = results_df[results_df['group'] == g]
    print(f"\n  {g} (N={len(subset)}):")
    print(f"  {'Ticker':7s} {'Name':15s} {'[-1,+1]':>9s} {'[0,0]':>9s} {'[0,+5]':>9s} {'[0,+10]':>9s}")
    for _, r in subset.iterrows():
        vals = [r.get(f'CAR_[{a:+d},{b:+d}]', np.nan) * 100 for a, b in event_windows]
        print(f"  {r['ticker']:7s} {r['name'][:15]:15s} " + " ".join(f"{v:+9.2f}" for v in vals))
    caar = subset['CAR_[-1,+1]'].mean() * 100
    print(f"  {'CAAR':7s} {'':15s} {caar:+9.2f}")

## 5. Results by Group

We now visualize the results for each group separately. The four-panel figure (CAAR with confidence band, AAR bars, individual CARs, cross-sectional distribution) is produced for the AI beneficiary group, the AI-disrupted group, and the control group. The visual comparison across groups is the core result of the study.

In [ ]:
# -- Four-panel figures by group --

def plot_group(ar_panel, group_name, car_df, color='#1f78b4'):
    ed = ar_panel.index.tolist()
    N = ar_panel.shape[1]
    aar = ar_panel.mean(axis=1)
    caar = aar.cumsum()
    aar_se = ar_panel.std(axis=1, ddof=1) / np.sqrt(N)
    caar_se = np.sqrt((aar_se ** 2).cumsum())

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    ax = axes[0, 0]
    ax.plot(ed, caar * 100, 'o-', color=color, linewidth=2.5, markersize=6)
    ax.fill_between(ed, (caar - 1.96 * caar_se) * 100,
                    (caar + 1.96 * caar_se) * 100, alpha=0.2, color=color, label='95% CI')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axvline(0, color='red', linewidth=1.5, linestyle='--', alpha=0.6, label='Event')
    ax.set_xlabel('Event Day ($\\tau$)'); ax.set_ylabel('CAAR (%)')
    ax.set_title(f'Panel A: CAAR — {group_name}')
    ax.legend(fontsize=9, frameon=True); ax.grid(True, alpha=0.2)

    ax = axes[0, 1]
    c = ['#d95f02' if v >= 0 else color for v in aar]
    ax.bar(ed, aar * 100, color=c, edgecolor='white', width=0.6)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axvline(0, color='red', linewidth=1.5, linestyle='--', alpha=0.6)
    ax.set_xlabel('Event Day ($\\tau$)'); ax.set_ylabel('AAR (%)')
    ax.set_title(f'Panel B: AAR — {group_name}'); ax.grid(True, alpha=0.2)

    ax = axes[1, 0]
    for col in ar_panel.columns:
        ax.plot(ed, ar_panel[col].cumsum() * 100, 'o-', alpha=0.5, linewidth=1, markersize=4)
    ax.plot(ed, caar * 100, 'k-', linewidth=2.5, label='CAAR')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axvline(0, color='red', linewidth=1.5, linestyle='--', alpha=0.6)
    ax.set_xlabel('Event Day ($\\tau$)'); ax.set_ylabel('CAR (%)')
    ax.set_title(f'Panel C: Individual CARs — {group_name}')
    ax.legend(fontsize=9, frameon=True); ax.grid(True, alpha=0.2)

    ax = axes[1, 1]
    sub = car_df[car_df['group'] == group_name.split(' ')[0]].sort_values('CAR_[-1,+1]')
    if len(sub) == 0:
        sub = car_df.sort_values('CAR_[-1,+1]')
    c2 = ['#d95f02' if v >= 0 else color for v in sub['CAR_[-1,+1]']]
    ax.barh(range(len(sub)), sub['CAR_[-1,+1]'] * 100, color=c2, edgecolor='white')
    ax.set_yticks(range(len(sub)))
    ax.set_yticklabels([f"{r['ticker']}" for _, r in sub.iterrows()], fontsize=9)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('CAR [-1,+1] (%)'); ax.set_title(f'Panel D: Cross-Section — {group_name}')
    ax.grid(True, alpha=0.2)

    plt.suptitle(f'ChatGPT Release: {group_name} (N={N})', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

# Produce figures
group_colors = {'Beneficiary': '#33a02c', 'Disrupted': '#e31a1c', 'Control': '#1f78b4'}
for g in ['Beneficiary', 'Disrupted', 'Control']:
    if g in ar_panels:
        group_label = f'{g} Group'
        plot_group(ar_panels[g], group_label, results_df, color=group_colors[g])

## 6. Formal Tests by Group

We apply the BMP test, the Corrado rank test (simplified), the cross-sectional t-test, and the generalized sign test to each group separately and to the pooled sample. The tests are conducted for the primary $[-1, +1]$ window.

In [ ]:
# -- Test battery by group --

def run_tests(ar_panel, model_params_dict, label):
    N = ar_panel.shape[1]
    cars = ar_panel.sum(axis=0)
    caar = cars.mean()
    se_cs = cars.std(ddof=1) / np.sqrt(N)
    t_cs = caar / se_cs if se_cs > 0 else 0
    p_cs = 2 * (1 - stats.t.cdf(abs(t_cs), df=max(N - 1, 1)))

    # BMP
    T_w = len(ar_panel.index)
    sar = ar_panel.copy()
    for tick in sar.columns:
        if tick in model_params_dict:
            sar[tick] = sar[tick] / model_params_dict[tick]
    csar = sar.sum(axis=0) / np.sqrt(T_w)
    csar_mean = csar.mean()
    csar_std = csar.std(ddof=1)
    t_bmp = csar_mean / (csar_std / np.sqrt(N)) if csar_std > 0 else 0
    p_bmp = 2 * (1 - stats.t.cdf(abs(t_bmp), df=max(N - 1, 1)))

    # Sign
    n_pos = (cars > 0).sum()
    t_sign = (n_pos - 0.5 * N) / np.sqrt(0.25 * N)
    p_sign = 2 * min(stats.binom.cdf(n_pos, N, 0.5),
                     1 - stats.binom.cdf(max(n_pos - 1, 0), N, 0.5))
    p_sign = min(p_sign, 1.0)

    # Rank (simplified: cross-sectional rank of CARs)
    rank_vals = cars.rank() / (N + 1) - 0.5
    r_mean = rank_vals.mean()
    r_std = rank_vals.std(ddof=0)
    t_rank = r_mean / (r_std / np.sqrt(N)) if r_std > 0 else 0
    p_rank = 2 * (1 - stats.norm.cdf(abs(t_rank)))

    return {
        'Group': label, 'N': N, 'CAAR (%)': caar * 100,
        'N+': n_pos,
        't_CS': t_cs, 'p_CS': p_cs,
        't_BMP': t_bmp, 'p_BMP': p_bmp,
        't_Sign': t_sign, 'p_Sign': p_sign,
        't_Rank': t_rank, 'p_Rank': p_rank,
    }

# Build sigma dict
sigma_dict = {r['ticker']: r['sigma'] for _, r in results_df.iterrows()}

test_rows = []
for g in ['Beneficiary', 'Disrupted', 'Control']:
    if g in ar_panels:
        test_rows.append(run_tests(ar_panels[g], sigma_dict, g))

# Pooled
ar_pooled = pd.concat([ar_panels[g] for g in ar_panels], axis=1)
test_rows.append(run_tests(ar_pooled, sigma_dict, 'Pooled'))

test_df = pd.DataFrame(test_rows)

print("Test Results by Group: Window [-1, +1]")
print("=" * 90)
print(f"\n  {'Group':12s} {'N':>3s} {'CAAR%':>7s} {'N+':>3s}  "
      f"{'t_CS':>7s} {'p':>6s}  {'t_BMP':>7s} {'p':>6s}  "
      f"{'t_Sign':>7s} {'p':>6s}  {'t_Rank':>7s} {'p':>6s}")
print(f"  {'-'*85}")
for _, r in test_df.iterrows():
    print(f"  {r['Group']:12s} {r['N']:3.0f} {r['CAAR (%)']:+7.2f} {r['N+']:3.0f}  "
          f"{r['t_CS']:+7.3f} {r['p_CS']:6.4f}  "
          f"{r['t_BMP']:+7.3f} {r['p_BMP']:6.4f}  "
          f"{r['t_Sign']:+7.3f} {r['p_Sign']:6.4f}  "
          f"{r['t_Rank']:+7.3f} {r['p_Rank']:6.4f}")

## 7. Cross-Sectional Regression

We regress the firm-level CAR on group dummies (with the control group as the omitted category), log market capitalization (proxied by the estimation-window average of the stock price times shares outstanding, or simply by the stock price level as a rough proxy), and the market model beta. The specification is:

$$
CAR_i = \gamma_0 + \gamma_1 D_{beneficiary,i} + \gamma_2 D_{disrupted,i} + \gamma_3 \beta_i + \eta_i
$$

The hypothesis is $\gamma_1 > 0$ (beneficiaries earn positive abnormal returns relative to the control group) and $\gamma_2 < 0$ (disrupted firms earn negative abnormal returns relative to the control group).

In [ ]:
# -- Cross-sectional regression --
reg_df = results_df.copy()
reg_df['CAR_pct'] = reg_df['CAR_[-1,+1]'] * 100
reg_df['D_beneficiary'] = (reg_df['group'] == 'Beneficiary').astype(int)
reg_df['D_disrupted'] = (reg_df['group'] == 'Disrupted').astype(int)

# Model 1: Group dummies only
y = reg_df['CAR_pct']
X1 = sm.add_constant(reg_df[['D_beneficiary', 'D_disrupted']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model 2: Group dummies + beta
X2 = sm.add_constant(reg_df[['D_beneficiary', 'D_disrupted', 'beta']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

from statsmodels.iolib.summary2 import summary_col
comparison = summary_col([model1, model2],
                         model_names=['(1) Groups', '(2) Groups + Beta'],
                         stars=True,
                         info_dict={'N': lambda x: f'{int(x.nobs)}',
                                    'R-squared': lambda x: f'{x.rsquared:.3f}',
                                    'Adj. R-sq': lambda x: f'{x.rsquared_adj:.3f}'})

print("Cross-Sectional Regression: CAR [-1,+1] (%)")
print("=" * 60)
print("Omitted category: Control group")
print("Standard errors: HC1")
print()
print(comparison)

print(f"\nInterpretation (Model 1):")
print(f"  Control group avg CAR: {model1.params['const']:+.2f}%")
print(f"  Beneficiary premium:   {model1.params['D_beneficiary']:+.2f}%")
print(f"  Disrupted discount:    {model1.params['D_disrupted']:+.2f}%")

## 8. Robustness Checks

### 8.1 Alternative Event Windows

We report the CAAR for each group across all four event windows. If the beneficiary effect strengthens at longer horizons, this is consistent with gradual market learning about AI implications (Eisfeldt et al., 2023). If the disrupted effect only appears at longer horizons, the market may have been slow to recognize the threat.

### 8.2 Excluding NVIDIA

NVIDIA is the dominant GPU supplier for AI workloads and may be an outlier among beneficiaries. We re-estimate the beneficiary CAAR with and without NVIDIA to assess its influence.

### 8.3 Placebo Test

We repeat the full analysis using a placebo event date of October 31, 2022 (one month before the actual release). Under the null of no effect on the placebo date, the CAAR should be statistically indistinguishable from zero for all groups.

In [ ]:
# -- Robustness --
print("Robustness Checks")
print("=" * 70)

# 8.1: CAAR across windows by group
print("\n  (a) CAAR (%) across event windows:")
print(f"  {'Group':12s}", end='')
for tau_a, tau_b in event_windows:
    print(f"  {'['+str(tau_a)+',+'+str(tau_b)+']':>10s}", end='')
print()
print(f"  {'-'*55}")
for g in ['Beneficiary', 'Disrupted', 'Control']:
    sub = results_df[results_df['group'] == g]
    print(f"  {g:12s}", end='')
    for tau_a, tau_b in event_windows:
        col = f'CAR_[{tau_a:+d},{tau_b:+d}]'
        caar = sub[col].mean() * 100
        print(f"  {caar:+10.2f}", end='')
    print()

# 8.2: Exclude NVIDIA
print(f"\n  (b) Beneficiary CAAR excluding NVIDIA:")
ben_all = results_df[results_df['group'] == 'Beneficiary']['CAR_[-1,+1]']
ben_no_nvda = results_df[(results_df['group'] == 'Beneficiary') &
                          (results_df['ticker'] != 'NVDA')]['CAR_[-1,+1]']
print(f"      With NVIDIA (N={len(ben_all)}):    CAAR = {ben_all.mean()*100:+.2f}%")
print(f"      Without NVIDIA (N={len(ben_no_nvda)}): CAAR = {ben_no_nvda.mean()*100:+.2f}%")
if len(ben_all) > 0:
    nvda_car = results_df[results_df['ticker'] == 'NVDA']['CAR_[-1,+1]'].values
    if len(nvda_car) > 0:
        print(f"      NVIDIA individual CAR: {nvda_car[0]*100:+.2f}%")

# 8.3: Placebo test
print(f"\n  (c) Placebo test (event date: October 31, 2022):")
placebo_date = pd.Timestamp('2022-10-31')
pidx = trading_days.get_indexer([placebo_date], method='ffill')[0]

placebo_cars = {}
for g in ['Beneficiary', 'Disrupted', 'Control']:
    sub_ticks = results_df[results_df['group'] == g]['ticker'].tolist()
    cars_g = []
    for tick in sub_ticks:
        r = results_df[results_df['ticker'] == tick].iloc[0]
        s = pidx - 1
        e = pidx + 1
        if s >= 0 and e < len(trading_days) and tick in firm_ret.columns:
            y_ev = firm_ret.iloc[s:e+1][tick].values
            x_ev = mkt_ret.iloc[s:e+1].values
            ar = y_ev - (r['alpha'] + r['beta'] * x_ev)
            cars_g.append(np.nansum(ar))
    if cars_g:
        placebo_cars[g] = np.array(cars_g)
        caar_p = np.mean(cars_g) * 100
        N_p = len(cars_g)
        se_p = np.std(cars_g, ddof=1) / np.sqrt(N_p) * 100
        t_p = caar_p / se_p if se_p > 0 else 0
        print(f"      {g:12s}: CAAR = {caar_p:+.2f}%, t = {t_p:+.3f} (N={N_p})")

## 9. Limitations

Several caveats apply to this study.

**Classification subjectivity.** The assignment of firms to the beneficiary, disrupted, and control groups is based on qualitative judgment about business model exposure to generative AI. Reasonable researchers could classify some firms differently. For example, Alphabet (GOOGL) is both a beneficiary (a leader in AI research with DeepMind and PaLM) and potentially disrupted (ChatGPT threatened Google Search's dominance). Our classification as a beneficiary reflects the ex ante assessment that Alphabet's AI capabilities would ultimately strengthen its competitive position, but the alternative classification is defensible.

**Partial anticipation.** GPT-3 had been available via API since June 2020, and DALL-E 2 was released in April 2022. Sophisticated investors may have already repriced AI exposure before November 30, 2022. If so, the measured abnormal returns understate the total value impact of large language models. The event study measures only the surprise component, which is the incremental information in the public release.

**Confounding events.** Although no major macroeconomic event occurred during the $[-1, +1]$ window, individual firms may have experienced idiosyncratic news (product launches, analyst reports, insider transactions) that coincides with the event window. With 20 firms, a thorough news screen for each firm-day observation is feasible and should be performed in a research-grade study.

**Small sample.** The sample of 20 firms (8 beneficiaries, 5 disrupted, 7 control) is small by event study standards. The test statistics have limited power, and the cross-sectional regression has few degrees of freedom. A larger sample, constructed from a systematic classification of all Russell 3000 firms by AI exposure, would provide more reliable estimates.

**Post-event drift.** The wider event windows ($[0, +5]$ and $[0, +10]$) may capture genuine post-event drift (gradual market learning) or may capture unrelated price movements. Distinguishing the two requires additional controls or the calendar-time portfolio approach of Session 7.

## 10. Conclusion and Course Summary

This capstone applied the complete event study toolkit to a timely and economically significant event: the public release of ChatGPT on November 30, 2022. The study illustrates how the methodology developed across all preceding sessions integrates into a coherent research workflow.

Session 1 provided the foundations: the event study logic, the abnormal return definition, and the five common pitfalls. Session 2 developed the normal return models. Session 3 addressed aggregation. Sessions 4 and 5 provided parametric and non-parametric test statistics. Session 6 introduced cross-sectional regressions. Session 7 extended the methodology to long horizons. Session 8 covered multi-country, intraday, confounding event, and conditional extensions. Session 8b developed the regression-based GARCH approach for single-entity, multiple-event settings. Session 9 organized the full workflow into a reproducible checklist.

The event study remains, more than five decades after Fama, Fisher, Jensen, and Roll (1969), one of the most widely used empirical methods in finance and economics. Its enduring appeal lies in its simplicity (the abnormal return is a clean measure of the market's valuation of new information), its flexibility (it applies to any event that can be dated), and its transparency (every design choice is visible and can be evaluated by the reader). The methodological refinements developed across this course, from the BMP test to the GARCH approach, ensure that the test statistics are reliable under realistic conditions, but the fundamental logic of the event study has not changed: measure what the market expected, measure what actually happened, and attribute the difference to the event.

**Further reading for the complete course:**

- Binder, J.J. (1998). The Event Study Methodology Since 1969. *Review of Quantitative Finance and Accounting*, 11(2), 111--137.
- Corrado, C.J. (2011). Event Studies: A Methodology Review. *Accounting and Finance*, 51(1), 207--234.
- Kothari, S.P. and Warner, J.B. (2007). Econometrics of Event Studies. In B.E. Eckbo (ed.), *Handbook of Empirical Corporate Finance*, Volume 1. Elsevier, Chapter 1.
- MacKinlay, A.C. (1997). Event Studies in Economics and Finance. *Journal of Economic Literature*, 35(1), 13--39.